# Chapter 18a — The Muon Optimizer: The Math

> Course: **llm.c — Zero to Hero**, companion to Chapter 18 (GPU AdamW).
> Builds on: Chapter 8 (AdamW & momentum), Chapter 18 (the GPU optimizer step).
>
> Audience: **freshman** — we assume you know what a gradient is, what a matrix is,
> and that training = "nudge the weights downhill". Everything else (singular values,
> orthogonal matrices) we build from scratch.

In 2024-2025, a new optimizer called **Muon** trained GPT-style models to the same loss
as AdamW in roughly **half the wall-clock time** on the famous *nanoGPT speedrun*, and the
Moonshot/Kimi team ([arXiv:2502.16982](https://arxiv.org/pdf/2502.16982)) showed it scales
to billion-parameter models. It is now one of the most talked-about optimizers in LLM training.

The name spells out the whole idea:

> **Muon = MomentUm Orthogonalized by Newton-schulz.**

This chapter unpacks each of those three words **as math you can run**. By the end you'll be
able to read the 8-line reference `zeropower_via_newtonschulz5` in `KellerJordan/Muon` and
explain every symbol.

### Learning objectives

By the end of this chapter you will be able to:

- Explain why treating a weight **matrix** as a flat list of numbers (what AdamW does) throws away structure.
- Define an **orthogonal / semi-orthogonal matrix** and the **singular values** of a matrix, intuitively.
- State what it means to **orthogonalize** an update: set all singular values to 1, i.e. replace `G` by `U Vᵀ`.
- Explain the **Newton-Schulz iteration**: a cheap polynomial that *approximately* orthogonalizes without ever computing an SVD.
- Write the full **Muon update rule** and say which parameters it applies to (and which fall back to AdamW).


## 1. The problem: AdamW treats a matrix as a bag of numbers

Recall AdamW from Chapter 8. For **every single weight** `w` it keeps a running mean `m`
and variance `v` of that weight's gradient, then steps:

$$w \leftarrow w - \text{lr}\cdot\frac{\hat m}{\sqrt{\hat v}+\epsilon}.$$

The key word is **every single weight, independently**. A `768 × 768` attention projection
has ~590k weights, and AdamW updates each one as if it lived alone on an island.

But that matrix is **not** a bag of 590k independent numbers — it's a **linear map**. It takes
a vector in and sends a vector out. Its rows and columns interact. When you ignore that
structure, something specific goes wrong, and we can see it with the **singular values**.


## 2. Singular values in 60 seconds

Any matrix `G` can be written as

$$G = U\,\Sigma\,V^{\top},$$

the **Singular Value Decomposition (SVD)**. Don't panic — here is all you need:

- `U` and `V` are **rotations** (orthogonal matrices: they spin/flip space but never stretch it).
- `Σ` (Sigma) is **diagonal**: its entries `σ₁ ≥ σ₂ ≥ … ≥ 0` are the **singular values** — the
  amounts of stretch along each principal direction.

So *every* matrix is "rotate → stretch each axis by some σ → rotate". The singular values are
the **stretch factors**. A matrix with one giant `σ₁` and the rest tiny is a *lopsided* map: it
blasts one direction and barely touches the others.

**That is exactly what momentum gradients look like.** A few directions dominate; the rest are
noise-sized. So a raw gradient step (and even AdamW's) keeps shoving the weights along the same
one or two loud directions, and learns the quiet directions painfully slowly.


In [ ]:
# See it: a random "update" matrix has a wildly uneven set of singular values.
import numpy as np
rng = np.random.default_rng(0)
G = rng.standard_normal((8, 6))          # pretend this is a momentum/gradient matrix
sigma = np.linalg.svd(G, compute_uv=False)
print("singular values (stretch factors):", np.round(sigma, 3))
print(f"biggest / smallest = {sigma.max()/sigma.min():.1f}x  -> very lopsided")


## 3. The fix: orthogonalize the update

What if we kept the **directions** of the update (the `U` and `V` rotations) but **flattened all
the stretch factors to 1**? Then every direction gets an equal-sized step — the loud directions
stop hogging the update and the quiet ones finally move.

Setting every `σ = 1` turns `G = U\,\Sigma\,V^{\top}` into

$$\text{orthogonalize}(G) = U\,I\,V^{\top} = U V^{\top}.$$

This `U Vᵀ` is the **nearest semi-orthogonal matrix** to `G` (the closest matrix whose singular
values are all 1). A **semi-orthogonal** matrix is one with `OᵀO = I` or `OOᵀ = I` — it preserves
lengths, treating all directions even-handedly. *That* is the "Orthogonalized" in Muon.

**Why is "equal in every direction" what we want?** Picture the loss surface as a **long, narrow
valley** — a half-pipe. *Across* the valley the walls are **steep**: a tiny move changes the loss a
lot, so the gradient is **large** there (a big singular value). *Along* the valley floor, the slope
toward the actual minimum is **gentle**, so the gradient is **tiny** (a small singular value). A raw
step follows the large direction, so it **bounces wall-to-wall across the valley** and barely creeps
along the floor toward the minimum — a big gradient means "it's steep here", **not** "this is the way
to the minimum". Flattening every singular value to 1 **shrinks the steep bouncing direction and
grows the gentle floor direction to the same size**, so the optimizer stops ricocheting and walks
straight down the valley. (We *see* this on a real loss surface in §3d.)

![Orthogonalizing compresses the singular-value spectrum toward 1](course/figures/fig_18a_orthogonalize.png)


In [ ]:
# Orthogonalize via the SVD: drop Sigma, keep U V^T.  All singular values become exactly 1.
U, S, Vt = np.linalg.svd(G, full_matrices=False)
O = U @ Vt
print("singular values of U V^T:", np.round(np.linalg.svd(O, compute_uv=False), 3))
print("all ~1.0 -> every direction now gets an equal-sized step")


## 3b. Concrete example — *what just happened?*

Abstract enough? Let's do it on a real **2×2** matrix, following the worked example in the
HuggingFace [*What just happened?*](https://huggingface.co/blog/onekq/muon-optimizer#what-just-happened)
walk-through. Take an update whose **two column vectors** are wildly out of balance:

$$G = \begin{bmatrix} 10.0 & 0.5 \\ 0.5 & 0.1 \end{bmatrix}.$$

Column 1 is **huge** and column 2 is **tiny and skewed** almost on top of it — one direction
completely dominates. Watch what orthogonalization does to those two vectors:

![Two skewed, unequal column vectors becoming perpendicular unit vectors](course/figures/anim_18a_orthogonalize.gif)

The animation shows the **two things** orthogonalization always does (straight from the blog):

1. **Makes the columns perpendicular** — they end up pointing in independent directions (a right angle).
2. **Normalizes their lengths** — every column ends up length 1.

In the language of 18a: the direction with **large** momentum (singular value 10) is scaled **down**,
the direction with **tiny** momentum (singular value 0.075) is scaled **up**, until both equal 1 —
*all directions become equally weighted in the update.*


In [ ]:
# The HuggingFace "what just happened" example, computed exactly.
G2 = np.array([[10.0, 0.5],
               [0.5,  0.1]])
print("singular values of G2 (how unequal the two directions are):",
      np.round(np.linalg.svd(G2, compute_uv=False), 4))    # ~[10.03, 0.075]  -> 134x lopsided!

U2, S2, Vt2 = np.linalg.svd(G2)
O2 = U2 @ Vt2                                               # orthogonalize: drop Sigma, keep U V^T
print("orthogonalized U V^T:\n", np.round(O2, 3))

c1, c2 = O2[:, 0], O2[:, 1]                                 # the two output column vectors
print("column 1 . column 2  (perpendicular => 0):", round(float(c1 @ c2), 6))
print("length of column 1, column 2 (both => 1):", round(float(np.linalg.norm(c1)), 4),
      round(float(np.linalg.norm(c2)), 4))
print("singular values after:", np.round(np.linalg.svd(O2, compute_uv=False), 4), " -> both 1")


So the `134×` imbalance between the two directions (`10.03` vs `0.075`) collapses to a perfect
`1 : 1`. The orthogonalized matrix here happens to be the clean identity frame — exactly the
*"perpendicular, equal length"* result the blog describes — because this particular `G` is symmetric.
For a general (non-symmetric) update you'd get a rotated orthonormal frame, but the two takeaways are
always the same: **perpendicular columns, unit lengths.**


## 3c. See the SVD itself — in 3D

The 2×2 picture above is the flat version. To really *see* what `G = U\,\Sigma\,V^{\top}` does, watch
it act on a **unit sphere in 3D**. SVD says every matrix is exactly three moves in a row:

1. **`Vᵀ` — rotate.** A rotation spins the sphere; a sphere looks the same, so nothing visible changes (yet).
2. **`Σ` — stretch.** Scale each axis by a singular value (here `3, 1.5, 0.6`). The sphere bulges into an **ellipsoid** whose semi-axis lengths *are* the singular values.
3. **`U` — rotate.** Spin the ellipsoid into its final orientation.

![SVD in 3D: a unit sphere is rotated, stretched into an ellipsoid by the singular values, then rotated again; orthogonalizing collapses it back to a sphere](course/figures/anim_18a_svd_3d.gif)

The ellipsoid is the whole story: a **lopsided** matrix (singular values `3` vs `0.6`) makes a
**stretched, cigar-shaped** ellipsoid — it pushes hard along one axis and barely along another.


**And orthogonalization, in this picture?** It sets **every singular value to 1**, which is the
same as deleting the `Σ` stretch — the ellipsoid snaps back into a **perfect sphere** (the last move
in the animation). A sphere reaches equally far in *every* direction: that is exactly "all directions
get an equal-sized step." The `134×` imbalance from the 2×2 example, and the `3 : 0.6` stretch here,
both collapse to a round, even sphere. That roundness is what Newton-Schulz is chasing — cheaply.


## 3d. See it on the loss landscape

We've watched orthogonalization fix one *update vector*. But why does that help **training**? To
see it, we visualize the **loss landscape** the optimizer is walking on, using the technique from
[Li et al., *Visualizing the Loss Landscape of Neural Nets* (NeurIPS 2018, arXiv:1712.09913)](https://arxiv.org/pdf/1712.09913)
and the [`tomgoldstein/loss-landscape`](https://github.com/tomgoldstein/loss-landscape) repo: pick a
2D plane through weight space and plot the loss as a contour map. (Li et al. slice along *random*
directions — **filter-normalized** so the scale is meaningful — for comparing sharpness; for showing
an optimizer's **path** they instead use the plane of the trajectory's top two **PCA** directions,
which is what we do here.)

The plane below cuts through the **same ill-conditioned least-squares problem** you'll train in
Chapter 18b. Its lopsided curvature — the matrix version of the lopsided singular values from §2 —
makes the loss surface a **long, narrow valley**:

![Loss landscape of an ill-conditioned problem: a long narrow valley, with SGD-momentum and Muon trajectories from the same start; Muon ends lower in the valley](course/figures/fig_18a_loss_landscape.png)


Both optimizers start at the same point (white square) and run the same number of steps, and — measured
in full weight space — they travel **almost exactly the same total distance** (path length `157` vs
`158`). The difference is *where that distance goes*. SGD-momentum keeps stepping hard along the steep
walls of the valley (the loud, high-curvature directions), so much of its travel is **side-to-side**,
and it stalls partway down. Muon **orthogonalizes** the step — equal size in every direction — so it
spends its budget moving *along the valley floor* toward the minimum, and ends at a loss of about
`0.18` versus SGD's `0.30` (a **1.7×** gap, the same margin you'll measure in 18b).

That is the whole payoff of §3 in one picture: equalizing the update's singular values turns a step
that **fights** an ill-conditioned valley into one that **flows down** it.


## 4. The catch: SVD is too slow for a training loop

`U Vᵀ` is exactly what we want — but computing an SVD is **expensive**, runs poorly on GPUs, and
usually demands `float32`. We do this for **every 2D weight, every single training step**. An SVD
per step would erase Muon's speed advantage.

We need orthogonalization that is:

- **cheap** — just matrix multiplies (which GPUs love),
- **GPU-friendly** — stable even in `bfloat16`,
- **good enough** — it does *not* have to be a perfect SVD.

Put plainly, the one thing we **give up is exactness**: the result's singular values will land
*near* 1, not exactly 1. We do **not** give up anything about the loss — we still minimize the same
loss; we just accept an *approximate* orthogonalization. For training that approximation is plenty
(from §3d, we only need the directions *roughly* evened out to flow down the valley), and in return
we get a method that is all matmuls and runs in `bfloat16`.

That last point is the trick. Enter **Newton-Schulz**.


## 5. Newton-Schulz: orthogonalize with only matmuls

Pick the **quintic** (degree-5) polynomial

$$p(x) = a\,x + b\,x^{3} + c\,x^{5},\qquad a=3.4445,\; b=-4.7750,\; c=2.0315,$$

and apply it **to the matrix** via the iteration (5 steps):

$$X \leftarrow a\,X + b\,(XX^{\top})X + c\,(XX^{\top})^{2}X.$$

Here is the beautiful part. Because of the `X = UΣVᵀ` structure, applying this matrix iteration
is *identical to* applying the scalar polynomial `p` **to each singular value** while leaving the
rotations `U, V` untouched. So we never touch an SVD — we just do a handful of matmuls — yet we are
secretly reshaping the singular values.

First we **normalize** `X` by its norm so all singular values land in `(0, 1]`; then we want `p`
to push every value in `(0, 1]` up toward 1.

![The Newton-Schulz polynomial flattens the singular values toward a band near 1](course/figures/fig_18a_newton_schulz.png)


**Honesty check — it lands *near* 1, not *exactly* 1.** Those coefficients were tuned for
**speed**, not for textbook convergence. After 5 steps the singular values sit in a band of
roughly `[0.7, 1.2]` — a 6× lopsided spectrum gets squashed to about 1.6×. That is *plenty* even
out the update directions for optimization, and it costs only ~5 iterations of matmuls. Trading a
perfect-but-slow SVD for a rough-but-fast polynomial is the entire engineering bet of Muon.


In [ ]:
# Newton-Schulz, the SAME math as KellerJordan/Muon's zeropower_via_newtonschulz5,
# but in plain numpy (the real one casts to bfloat16 on the GPU).
a, b, c = 3.4445, -4.7750, 2.0315

def newton_schulz(G, steps=5):
    X = G / (np.linalg.norm(G) + 1e-7)        # normalize so singular values <= 1
    transposed = X.shape[-2] > X.shape[-1]    # work on the "wide" orientation for stability
    if transposed:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    if transposed:
        X = X.T
    return X

NS = newton_schulz(G, steps=5)
print("singular values after Newton-Schulz:", np.round(np.linalg.svd(NS, compute_uv=False), 3))
print("compare raw spread vs NS spread:")
s_ns = np.linalg.svd(NS, compute_uv=False)
print(f"  raw: {sigma.max()/sigma.min():.1f}x   ->   NS: {s_ns.max()/s_ns.min():.1f}x  (much flatter, with NO svd call)")


**Why a *quintic*, and why `XXᵀ`?** The polynomial only uses **odd** powers of `X`
(`x`, `x³`, `x⁵`). An odd power like `x⁵` written with matrices is `X(XᵀX)(XᵀX)` — it keeps the
same `U, V` rotations and only re-scales the singular values, which is exactly what we want. Even
powers would mix the rotations and ruin the trick. The single most expensive operation in the
whole iteration is therefore the symmetric product `A = X Xᵀ` — **remember that line, Chapter 18c
makes it fast on CUDA.**


## 6. The full Muon update rule

Put the three words together. For one 2D weight matrix `W` with gradient `G`:

```
1. momentum:        M <- b*M + (1-b)*G            # SGD-momentum, b = 0.95   ("MomentUm")
2. (nesterov) look: U <- b*M + (1-b)*G            # lerp toward the fresh gradient
3. orthogonalize:   O <- NewtonSchulz(U, 5 steps) # ("Orthogonalized by Newton-schulz")
4. shape scale:     O <- O * sqrt(max(1, rows/cols))
5. weight update:   W <- W - lr * O               # (with decoupled weight decay)
```

Step 4 is a small bookkeeping fix: orthogonalization makes every singular value ≈ 1, so the
update's typical element size depends only on the matrix **shape**; the `sqrt(max(1, rows/cols))`
factor rescales it so Muon's effective step size behaves consistently across differently-shaped
layers (this keeps its update RMS comparable to AdamW's, so you can reuse familiar learning rates).

Written compactly, the orthogonalized direction is

$$O \;=\; \text{NewtonSchulz}\!\big(\beta M + (1-\beta)G\big)\cdot\sqrt{\max\!\big(1,\tfrac{\text{rows}}{\text{cols}}\big)}.$$


## 6a. Step 4 up close — why the shape scale?

Step 4 is the one line that looks arbitrary, so let's make it concrete. After Newton-Schulz every
singular value ≈ 1, and a matrix with `n` singular values all equal to 1 has total "energy" (the sum
of every entry squared) of exactly `n`. So the **typical entry size** — the root-mean-square (RMS) of
the update we actually apply — is

$$\text{RMS} = \sqrt{\dfrac{\text{energy}}{\#\text{entries}}} = \sqrt{\dfrac{n}{\text{rows}\times\text{cols}}}.$$

Compare two orthogonalized updates of **different shape**:

| matrix | # singular values `n` | energy | # entries | RMS (typical step) |
|---|---|---|---|---|
| square `128 × 128` | 128 | 128 | 16384 | `√(128/16384) ≈ 0.088` |
| tall `256 × 128` | 128 | 128 | 32768 | `√(128/32768) ≈ 0.0625` |

Both are "perfectly orthogonalized", yet the **tall** one's typical step is *smaller* — its extra
rows dilute the same energy over twice as many entries. With a single global learning rate the tall
layer would secretly take smaller steps than the square one. Step 4 fixes exactly this: multiply by
`sqrt(max(1, rows/cols))`. For the tall matrix that is `sqrt(256/128) = √2 ≈ 1.41`, and
`0.0625 × 1.41 ≈ 0.088` — back in line with the square layer. So **the shape scale makes every
layer's typical step the same size regardless of shape**, which is why one learning rate (borrowed
straight from AdamW) works across all of them.


In [ ]:
# Step 4, measured: orthogonalized updates of different shapes have different typical step sizes.
def rms(M): return float(np.sqrt((M ** 2).mean()))     # root-mean-square of the entries

for rows, cols in [(128, 128), (256, 128)]:
    G_shape = rng.standard_normal((rows, cols))
    O = newton_schulz(G_shape, 5)                       # singular values ~1, NO shape scale yet
    scale = max(1, rows / cols) ** 0.5
    print(f"{rows}x{cols}:  RMS before scale = {rms(O):.4f}   "
          f"x sqrt({rows}/{cols})={scale:.2f}   ->  RMS after = {rms(O) * scale:.4f}")
print("after the shape scale, both shapes land at the SAME typical step size -> one lr fits all")


In [ ]:
# The complete Muon direction for a single matrix, in ~6 lines of numpy.
def muon_update(grad, momentum, beta=0.95, ns_steps=5, nesterov=True):
    momentum[:] = beta * momentum + (1 - beta) * grad          # update momentum buffer (in place)
    look = beta * momentum + (1 - beta) * grad if nesterov else momentum
    o = newton_schulz(look, steps=ns_steps)
    o = o * (max(1, o.shape[-2] / o.shape[-1]) ** 0.5)          # shape scale
    return o

W = rng.standard_normal((256, 128))      # a hidden weight matrix
M = np.zeros_like(W)                     # momentum buffer starts at zero
g = rng.standard_normal((256, 128))      # this step's gradient

direction = muon_update(g, M)
lr = 0.02
W_new = W - lr * direction
print("update direction shape:", direction.shape)
sv = np.round(np.linalg.svd(direction, compute_uv=False)[:5], 3)
print("its singular values are all roughly EQUAL (flat):", sv, "...")
print("(here ~1.6, not ~1, because the sqrt(256/128)=1.41 shape-scale rescales them — but still flat)")
print(f"max weight change this step: {np.abs(W_new - W).max():.4f}")


## 7. Which parameters does Muon touch?

Muon's whole premise is the **2D matrix structure**, so it is used **only for the hidden weight
matrices** (the attention and MLP projections). Everything else falls back to plain **AdamW**:

| Parameter | Optimizer | Why |
|---|---|---|
| Hidden weight matrices (`ndim ≥ 2`) | **Muon** | They are linear maps — orthogonalizing helps. |
| Embeddings & output / classifier head | AdamW | Row-wise "lookup" tables, not balanced maps; Muon hurts them. |
| LayerNorm gains, biases, scalars (`ndim < 2`) | AdamW | 1D — "orthogonalize" is meaningless. |

So real training runs a **hybrid**: Muon on the big matmul weights, AdamW on everything else.
Chapter 18b builds exactly this hybrid and trains a tiny network with it.

```mermaid
flowchart TD
  P["a parameter tensor"] --> Q{"ndim >= 2 AND a hidden weight?"}
  Q -- "yes (attn / mlp matrices)" --> M["Muon: momentum -> Newton-Schulz -> scaled step"]
  Q -- "no (embeddings, head, norms, biases)" --> A["AdamW (Chapter 8 / 18)"]
```


## 7a. Concrete example — *why* embeddings stay on AdamW

The embedding table is `vocab × width` (e.g. `50257 × 768`) — technically 2D, yet it stays on AdamW.
Two reasons, both easiest to see on a toy **6-token** vocabulary (a `6 × 4` table, one row per token):

**Reason 1 — each row is private and used row-wise.** A forward pass reads only the rows for the
tokens in the batch, so the right update is **row-wise**: each token's row should move on its own,
independently of the others. Orthogonalization does the opposite — it **mixes all rows together** to
make them mutually orthogonal and equal-sized, coupling rows that have nothing to do with each other.
For an attention projection that coupling is the point (all rows act jointly on every token); for a
lookup table it is exactly wrong.

**Reason 2 — frequent vs rare tokens need different step sizes.** "the" appears in ~5% of all tokens,
so its row accumulates a **large** gradient; "defenestrate" maybe once in 10M, so its row stays
**tiny**. That imbalance is *correct* — AdamW's per-parameter `1/√v` keeps each token's step suited to
its own history. Orthogonalization **equalizes every row's update size**: it shrinks the loud "the"
row and amplifies the quiet rare rows until all rows step by the same amount, erasing the
specialization. The cell below shows one big row and five tiny rows getting flattened to one size.


In [ ]:
# Reason 2, measured: orthogonalization EQUALIZES per-row update sizes, destroying the natural
# "frequent token = big gradient, rare token = tiny gradient" structure an embedding relies on.
vocab, width = 6, 4
emb_grad = np.zeros((vocab, width))
emb_grad[0]  = 8.0 * rng.standard_normal(width)            # "the": frequent -> large gradient row
emb_grad[1:] = 0.05 * rng.standard_normal((vocab - 1, width))  # rare tokens -> tiny gradient rows
row_norm = lambda M: np.round(np.linalg.norm(M, axis=1), 3)
print("row sizes BEFORE (one loud row, five quiet):", row_norm(emb_grad))

O_emb = newton_schulz(emb_grad, 5)
print("row sizes AFTER orthogonalization (all ~equal):", row_norm(O_emb))
print("the loud 'the' row was shrunk and the quiet rare rows amplified -> per-token sizing destroyed")


## 8. Why is it faster? The FLOP budget

A natural worry: doesn't 5 Newton-Schulz iterations *add* work? Yes — but a tiny amount. The extra
cost is bounded by about `T*m / B`, where `T = 5` (iterations), `m` = model width, and `B` = tokens
per batch. From the Muon write-up:

- nanoGPT: `5 × 768 / 524288 ≈ 0.7%` extra FLOPs.
- Llama-405B-scale: `5 × 16384 / 16e6 ≈ 0.5%` extra FLOPs.

You pay **well under 1%** more compute per step, and in exchange each step makes more progress, so
you reach the target loss in fewer steps / less wall-clock time. That is the trade Muon wins.


## 9. Exercises

Predict each answer **before** running. Solutions are collapsed — click to reveal after you commit.


### Exercise 1 — what does orthogonalization do to a rank-deficient update?

Build a matrix whose singular values are very lopsided (e.g. `[5, 5, 0.01, 0.01, ...]`) by
constructing `G = U @ diag(s) @ V`. **Predict:** after Newton-Schulz, will the tiny directions stay
tiny, or be lifted up toward 1? Verify by printing the singular values before and after.


In [ ]:
# Your attempt: build a lopsided G, run newton_schulz, compare singular values.
rng2 = np.random.default_rng(7)
Q1, _ = np.linalg.qr(rng2.standard_normal((6, 6)))   # a random rotation U
Q2, _ = np.linalg.qr(rng2.standard_normal((6, 6)))   # a random rotation V
s = np.array([5.0, 5.0, 0.01, 0.01, 0.01, 0.01])
G_lop = Q1 @ np.diag(s) @ Q2
print("before:", np.round(np.linalg.svd(G_lop, compute_uv=False), 3))
# TODO: print the singular values of newton_schulz(G_lop, 5) and compare


<details>
<summary>▶ Show solution</summary>

```python
out = newton_schulz(G_lop, 5)
print("after: ", np.round(np.linalg.svd(out, compute_uv=False), 3))
# The tiny 0.01 directions are LIFTED up into the ~[0.7, 1.2] band — that is the whole point.
# Orthogonalization rescues the "quiet" directions that a raw gradient step would ignore for ages.
```

The quiet directions get amplified to roughly the same size as the loud ones. AdamW would keep
crawling along them for thousands of steps; Muon equalizes them in one orthogonalization.
</details>


### Exercise 2 — how many Newton-Schulz steps are "enough"?

For the lopsided `G_lop` above, run `newton_schulz` with `steps = 0, 1, 2, 3, 5, 10` and print the
**spread** (max σ / min σ) each time. **Predict:** does the spread keep shrinking forever, or
plateau? Where does 5 steps sit?


In [ ]:
# Your attempt: loop over step counts and print the singular-value spread.
for k in [0, 1, 2, 3, 5, 10]:
    pass  # TODO: out = newton_schulz(G_lop, k); s = svd values; print k and s.max()/s.min()


<details>
<summary>▶ Show solution</summary>

```python
for k in [0, 1, 2, 3, 5, 10]:
    s = np.linalg.svd(newton_schulz(G_lop, k), compute_uv=False)
    print(f"steps={k:2d}  spread = {s.max()/s.min():.2f}x")
# The spread drops fast for the first few steps, then plateaus: the polynomial cannot do
# better than its tuned band. ~5 steps captures almost all the benefit — more is wasted compute.
```

This is why the reference default is `ns_steps = 5`: it sits right at the knee of the curve.
</details>


### Exercise 3 — why not orthogonalize the embedding table?

The token embedding is a `vocab × width` matrix (e.g. `50257 × 768`). Argue in 2-3 sentences why
forcing all its singular values to ≈1 is a **bad** idea, unlike for an attention projection.
(Hint: each *row* is one token's vector, and only a few tokens appear in any batch.)


<details>
<summary>▶ Show solution</summary>

An embedding is a **lookup table**, not a balanced linear map: a forward pass reads only the
handful of rows for the tokens actually in the batch, so its update is naturally **sparse and
row-wise**. Orthogonalization mixes information *across all rows* to equalize singular values,
smearing the few rows that should change into the millions that shouldn't — destroying the
per-token specialization. Frequent vs rare tokens also deserve very different effective step
sizes, which is exactly AdamW's per-parameter adaptivity. So embeddings (and the output head)
stay on AdamW.
</details>


## Further Reading

**Source of truth**

- [Keller Jordan — *Muon: An optimizer for the hidden layers of neural networks*](https://kellerjordan.github.io/posts/muon/) — the canonical write-up; the orthogonalization argument and the FLOP-overhead numbers used above.
- [`KellerJordan/Muon`](https://github.com/KellerJordan/Muon) (`muon.py`) — the reference `zeropower_via_newtonschulz5` and the `muon_update` we ported to numpy here.
- [*Muon is Scalable for LLM Training* (Moonshot/Kimi), arXiv:2502.16982](https://arxiv.org/pdf/2502.16982) — scaling Muon to billion-parameter models, the weight-decay and update-RMS analysis behind the shape scale.

**Going deeper**

- [HuggingFace blog — *The Muon optimizer*](https://huggingface.co/blog/onekq/muon-optimizer) — a gentle second pass over the same ideas.
- [Polar decomposition (Wikipedia)](https://en.wikipedia.org/wiki/Polar_decomposition) — the classical orthogonal factor `U Vᵀ` that Newton-Schulz approximates.
- [Li et al., *Visualizing the Loss Landscape of Neural Nets*, arXiv:1712.09913](https://arxiv.org/pdf/1712.09913) + [`tomgoldstein/loss-landscape`](https://github.com/tomgoldstein/loss-landscape) — the filter-normalized / PCA-trajectory visualization used for the loss-landscape figure in §3d.

**Sibling chapters**

- **Chapter 18b** — implement the full hybrid (Muon + AdamW) and train a tiny network with it.
- **Chapter 18c** — make Newton-Schulz fast on CUDA/C++ (the `X Xᵀ` symmetric-matmul trick).


## Recap

- AdamW updates each weight **independently**, ignoring that a weight matrix is a **linear map**.
- The **singular values** of an update are its stretch factors; momentum updates are **lopsided** (a few huge, many tiny).
- **Orthogonalizing** sets every singular value to 1 — replace `G` by `U Vᵀ` — so all directions get an equal step.
- A true SVD is too slow, so Muon uses the **Newton-Schulz** quintic (`a=3.4445, b=-4.7750, c=2.0315`, 5 steps): only matmuls, runs in bf16, lands singular values in a band **near** 1 (not exactly 1 — tuned for speed).
- **Muon = momentum → Newton-Schulz orthogonalize → shape-scale → step**, applied to **2D hidden weights only**; embeddings/head/norms stay on **AdamW**. Cost: **< 1%** extra FLOPs.

### What's next

**Chapter 18b — Implementing Muon.** We turn this math into a working optimizer in numpy, build
the **hybrid Muon+AdamW** parameter router, and watch Muon out-converge plain SGD/AdamW on a small
problem — then map every line back to the reference `KellerJordan/Muon`.
